# Explainable GeoAI Demo for Urban Accessibility

This notebook is a **public portfolio demonstration** using synthetic, non-sensitive data.

It illustrates how social and spatial predictors can be combined with geographically structured validation to model experienced accessibility to urban services. It does not contain raw dissertation data.


## 1. Imports


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", None)


## 2. Load the synthetic dataset


In [ ]:
DATA_PATH = Path("../data/synthetic_accessibility_demo.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


## 3. Define predictors and target


In [ ]:
predictors = [
    "gender_indicator",
    "caregiving",
    "mobility_limitation",
    "income_index",
    "safety_score",
    "mobility_score",
    "service_quality",
    "nearest_service_m",
    "opportunity_1000m",
    "potential_access_index",
]

X = df[predictors]
y = df["experienced_access"]
groups = df["district"]
X.describe().T


## 4. District-held-out cross-validation

`GroupKFold` prevents observations from the same district from appearing in both the training and validation fold.


In [ ]:
model = GradientBoostingRegressor(random_state=42)
cv = GroupKFold(n_splits=5)

oof_pred = cross_val_predict(
    model,
    X,
    y,
    groups=groups,
    cv=cv
)

r2 = r2_score(y, oof_pred)
mae = mean_absolute_error(y, oof_pred)
rmse = mean_squared_error(y, oof_pred) ** 0.5

print(f"OOF R²:   {r2:.3f}")
print(f"OOF MAE:  {mae:.3f}")
print(f"OOF RMSE: {rmse:.3f}")


## 5. Observed vs predicted accessibility


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y, oof_pred, alpha=0.7)
lims = [min(y.min(), oof_pred.min()), max(y.max(), oof_pred.max())]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Observed experienced access")
plt.ylabel("Out-of-fold prediction")
plt.title("Observed vs predicted accessibility")
plt.tight_layout()
plt.show()


## 6. Fit the final model and inspect feature importance


In [ ]:
model.fit(X, y)

importance = (
    pd.Series(model.feature_importances_, index=predictors)
    .sort_values(ascending=False)
)

importance


In [ ]:
plt.figure(figsize=(7, 5))
importance.sort_values().plot(kind="barh")
plt.xlabel("Model feature importance")
plt.title("Feature importance in the synthetic demo")
plt.tight_layout()
plt.show()


## 7. Optional SHAP explanation

If `shap` is installed, this section calculates SHAP values for the fitted model.


In [ ]:
try:
    import shap

    explainer = shap.Explainer(model, X)
    shap_values = explainer(X)
    shap.plots.bar(shap_values, max_display=10)
except ImportError:
    print("SHAP is not installed. Run: pip install shap")
except Exception as exc:
    print("SHAP could not run in this environment:", exc)


## Interpretation note

This notebook demonstrates a reproducible workflow rather than a substantive empirical result. Because the dataset is synthetic, predictive performance and feature importance should not be interpreted as findings about Tehran or any real population.
